# 07 — A/B Testing for Models

## What
Model A/B testing routes a fraction of production traffic to a challenger model while the champion handles the rest. Outcomes are compared statistically before deciding which model wins. Unlike offline evaluation, A/B tests capture real-world user behaviour, system effects, and label distribution under the live data distribution.

## Why
Offline metrics (AUC, F1) optimise for the training distribution. Production traffic follows its own distribution, shifted by seasonality, demographic changes, and feedback loops. A model that wins on held-out data may lose in production — A/B testing catches this before a full rollout.

## Community context
Two-sample z-tests and Mann-Whitney U are the workhorses. Sequential testing (SPRT, always-valid inference) lets you stop early when one model clearly dominates — reducing the cost of running a bad challenger. Interleaving (showing results from both models in one request) is used in search/recommendation to get higher statistical power with less traffic.

In [ ]:
# Model A/B test harness with statistical significance testing
import numpy as np
from scipy import stats

class ABTestHarness:
    def __init__(self, control_name, treatment_name, traffic_split=0.1, random_seed=42):
        self.control = control_name
        self.treatment = treatment_name
        self.split = traffic_split   # fraction of traffic to treatment
        self.rng = np.random.default_rng(random_seed)
        self.logs = {'control': [], 'treatment': []}

    def route(self, request_id):
        """Deterministic routing based on request_id hash"""
        h = hash(request_id) % 1000
        return 'treatment' if h < self.split * 1000 else 'control'

    def log_outcome(self, request_id, outcome):
        bucket = self.route(request_id)
        self.logs[bucket].append(outcome)

    def analyze(self, alpha=0.05):
        ctrl = np.array(self.logs['control'])
        trt  = np.array(self.logs['treatment'])
        if len(ctrl) == 0 or len(trt) == 0:
            return {'error': 'No data'}
        ctrl_mean = ctrl.mean()
        trt_mean  = trt.mean()
        # Two-sided t-test (for continuous outcomes)
        t_stat, p_val = stats.ttest_ind(ctrl, trt)
        # Effect size (Cohen's d)
        pooled_std = np.sqrt((ctrl.std()**2 + trt.std()**2) / 2)
        cohens_d = (trt_mean - ctrl_mean) / (pooled_std + 1e-10)
        significant = p_val < alpha
        return {
            'control_n': len(ctrl), 'treatment_n': len(trt),
            'control_mean': round(ctrl_mean, 4),
            'treatment_mean': round(trt_mean, 4),
            'lift_pct': round(100*(trt_mean-ctrl_mean)/ctrl_mean, 2),
            'p_value': round(p_val, 4),
            'cohens_d': round(cohens_d, 3),
            'significant': significant,
            'recommendation': 'PROMOTE treatment' if (significant and trt_mean > ctrl_mean) else
                             ('REJECT treatment' if (significant and trt_mean < ctrl_mean) else
                              'INCONCLUSIVE - need more data')
        }

# Simulate: treatment model has genuinely higher precision on fraud detection
np.random.seed(42)
ab = ABTestHarness('model_v1', 'model_v2', traffic_split=0.1)

# Simulate 10000 requests
for i in range(10000):
    req_id = f'req_{i}'
    bucket = ab.route(req_id)
    if bucket == 'control':
        # Control: 72% precision on fraud labels
        outcome = np.random.binomial(1, 0.72)
    else:
        # Treatment: 78% precision
        outcome = np.random.binomial(1, 0.78)
    ab.log_outcome(req_id, outcome)

result = ab.analyze()
print('=== A/B Test Analysis ===')
for k, v in result.items():
    print(f'  {k:<22}: {v}')

## Sequential Testing (SPRT)

Fixed-horizon tests waste time when a winner emerges early. Sequential Probability Ratio Testing (SPRT) lets you stop as soon as evidence is strong enough — saving both time and bad-model traffic.

In [ ]:
# Simplified SPRT for binary outcomes
def sprt_test(control_outcomes, treatment_outcomes,
              h0_lift=0.0, h1_lift=0.05, alpha=0.05, beta=0.20):
    """
    H0: treatment effect = h0_lift
    H1: treatment effect = h1_lift
    Returns the step at which we could have stopped.
    """
    A = (1 - beta) / alpha     # upper boundary -> reject H0 (treatment wins)
    B = beta / (1 - alpha)     # lower boundary -> accept H0 (no effect)
    p0 = np.mean(control_outcomes)
    p1_h1 = p0 + h1_lift
    log_lambda = 0.0
    for step, (c, t) in enumerate(zip(control_outcomes, treatment_outcomes)):
        # Likelihood ratio for this observation pair
        if t == 1:
            log_lambda += np.log(p1_h1 / p0 + 1e-10)
        else:
            log_lambda += np.log((1-p1_h1) / (1-p0+1e-10) + 1e-10)
        lam = np.exp(log_lambda)
        if lam >= A:
            return step, 'REJECT H0 (treatment wins)', lam
        if lam <= B:
            return step, 'ACCEPT H0 (no effect)', lam
    return step, 'INCONCLUSIVE', np.exp(log_lambda)

np.random.seed(0)
ctrl_out = np.random.binomial(1, 0.72, 2000)
trt_out  = np.random.binomial(1, 0.78, 2000)
step, decision, lam = sprt_test(ctrl_out, trt_out)
print(f'SPRT stopped at step {step} ({100*step/2000:.0f}% of max samples)')
print(f'Decision: {decision} (lambda={lam:.2f})')